# PassCaptcha - Train YOLOv8 Cho reCAPTCHA Grid
Fine-tune mô hình YOLOv8n (Nano) nhận diện các vật thể reCAPTCHA (buses, traffic lights, bicycles, hydrants).
- **Backend:** Ultralytics YOLOv8 PyTorch
- **Tối ưu:** Export ONNX INT8 chạy trên CPU < 80ms.

In [ ]:
!pip install ultralytics onnx onnxruntime --quiet

In [ ]:
from ultralytics import YOLO
import torch

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

### 1. Chuẩn Bị File Cấu Hình data.yaml Cho reCAPTCHA Dataset

In [ ]:
import yaml

# File cấu hình các class phổ biến nhất trong reCAPTCHA v2
dataset_config = {
    'path': '/kaggle/working/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'bicycle',
        1: 'motorcycle',
        2: 'bus',
        3: 'car',
        4: 'traffic_light',
        5: 'fire_hydrant',
        6: 'crosswalk',
        7: 'chimney'
    }
}

with open('recaptcha_data.yaml', 'w') as f:
    yaml.dump(dataset_config, f)
print("Đã tạo recaptcha_data.yaml")

### 2. Fine-tune YOLOv8 Nano Bằng GPU T4

In [ ]:
# Load model gốc COCO pretrained
model = YOLO('yolov8n.pt')

# Bắt đầu train (Ví dụ 30-50 epochs trên Kaggle T4 chỉ mất ~20 phút)
results = model.train(
    data='coco8.yaml', # Thay bằng đường dẫn dataset reCAPTCHA thực tế của bạn
    epochs=30,
    imgsz=320,
    batch=32,
    device=0 if torch.cuda.is_available() else 'cpu'
)

### 3. Xuất File ONNX Cho PassCaptcha

In [ ]:
# Export sang định dạng ONNX
success = model.export(format='onnx', imgsz=320, dynamic=True, simplify=True)
print("[SUCCESS] File ONNX đã sẵn sàng để tải về máy!")